# Detailed Site Information

Load and inspect the HEMS-RBSA base site data.

In [1]:
from pathlib import Path
import sys

import pandas as pd

In [2]:
data_candidates = [
    Path("../src/data/HEMS_RBSA/HEMS_RBSA_BASE.csv"),
    Path("validation/src/data/HEMS_RBSA/HEMS_RBSA_BASE.csv"),
]
base_csv_path = next(path for path in data_candidates if path.exists())

site_info = pd.read_csv(base_csv_path, dtype=str)

## Filter RBSA II Raw Files

Select the RBSA II `SiteID` values, match them against `CK_SiteID` in each raw CSV, and save nonempty filtered files under `RBSA_II/HEMS`.

In [3]:
rbsa_ii_site_ids = set(
    site_info.loc[site_info["RBSA Data Source"].str.strip().eq("RBSA II"), "SiteID"]
    .dropna()
    .str.strip()
)

rbsa_ii_dir = base_csv_path.parent / "RBSA_II"
raw_dir = rbsa_ii_dir / "raw"
hems_dir = rbsa_ii_dir / "HEMS"
hems_dir.mkdir(parents=True, exist_ok=True)

print(f"RBSA II SiteIDs: {len(rbsa_ii_site_ids):,}")
print(f"Raw directory:  {raw_dir}")
print(f"Output directory: {hems_dir}")

RBSA II SiteIDs: 222
Raw directory:  ..\src\data\HEMS_RBSA\RBSA_II\raw
Output directory: ..\src\data\HEMS_RBSA\RBSA_II\HEMS


In [ ]:
filter_results = []

for raw_path in sorted(raw_dir.glob("*.csv")):
    raw_data = pd.read_csv(raw_path, dtype=str)

    if "CK_SiteID" not in raw_data.columns:
        filter_results.append({
            "file": raw_path.name,
            "status": "skipped: no CK_SiteID",
            "raw_rows": len(raw_data),
            "matched_rows": 0,
        })
        continue

    matched = raw_data[raw_data["CK_SiteID"].str.strip().isin(rbsa_ii_site_ids)].copy()
    if matched.empty:
        status = "no matches"
    else:
        matched.to_csv(hems_dir / raw_path.name, index=False)
        status = "saved"

    filter_results.append({
        "file": raw_path.name,
        "status": status,
        "raw_rows": len(raw_data),
        "matched_rows": len(matched),
    })

filter_summary = pd.DataFrame(filter_results)

,file,status,raw_rows,matched_rows
0,Appliance_AudioEquipment.csv,saved,1387,181
1,Appliance_ClothesDryer.csv,saved,1852,215
2,Appliance_ClothesWasher.csv,saved,1880,218
3,Appliance_Computer.csv,saved,2131,258
4,Appliance_Dehumidifier.csv,saved,41,8
5,Appliance_Dishwasher.csv,saved,1592,199
6,Appliance_GameConsole.csv,saved,794,88
7,Appliance_LargeUnusualLoad.csv,saved,344,56
8,Appliance_PowerStrip.csv,saved,1936,282
9,Appliance_Refrigerator_Freezer.csv,saved,3153,368


In [ ]:
filter_summary.groupby("status", dropna=False).agg(
    files=("file", "count"),
    matched_rows=("matched_rows", "sum"),
)

## Filter RBSA III Raw Files

Select the RBSA III `SiteID` values, match them against `SiteID` in each raw CSV, and save nonempty filtered files under `RBSA_III/HEMS`.

In [5]:
rbsa_iii_site_ids = set(
    site_info.loc[site_info["RBSA Data Source"].str.strip().eq("RBSA III"), "SiteID"]
    .dropna()
    .str.strip()
)

rbsa_iii_dir = base_csv_path.parent / "RBSA_III"
rbsa_iii_raw_dir = rbsa_iii_dir / "raw"
rbsa_iii_hems_dir = rbsa_iii_dir / "HEMS"
rbsa_iii_hems_dir.mkdir(parents=True, exist_ok=True)

print(f"RBSA III SiteIDs: {len(rbsa_iii_site_ids):,}")
print(f"Raw directory:  {rbsa_iii_raw_dir}")
print(f"Output directory: {rbsa_iii_hems_dir}")

RBSA III SiteIDs: 133
Raw directory:  ..\src\data\HEMS_RBSA\RBSA_III\raw
Output directory: ..\src\data\HEMS_RBSA\RBSA_III\HEMS


In [8]:
rbsa_iii_filter_results = []

for raw_path in sorted(rbsa_iii_raw_dir.glob("*.csv")):
    raw_data = pd.read_csv(raw_path, dtype=str)

    if "SiteID" not in raw_data.columns:
        rbsa_iii_filter_results.append({
            "file": raw_path.name,
            "status": "skipped: no SiteID",
            "raw_rows": len(raw_data),
            "matched_rows": 0,
        })
        continue

    matched = raw_data[raw_data["SiteID"].str.strip().isin(rbsa_iii_site_ids)].copy()
    if matched.empty:
        status = "no matches"
    else:
        matched.to_csv(rbsa_iii_hems_dir / raw_path.name, index=False)
        status = "saved"

    rbsa_iii_filter_results.append({
        "file": raw_path.name,
        "status": status,
        "raw_rows": len(raw_data),
        "matched_rows": len(matched),
    })

rbsa_iii_filter_summary = pd.DataFrame(rbsa_iii_filter_results)

In [9]:
rbsa_iii_filter_summary.groupby("status", dropna=False).agg(
    files=("file", "count"),
    matched_rows=("matched_rows", "sum"),
)

,files,matched_rows
status,,
saved,54,19126
skipped: no SiteID,7,0


## RBSA III Raw Data Inventory

Create one combined schema inventory describing the information available in every RBSA III raw CSV. Each row represents one source column rather than one site.

In [3]:
inventory_rows = []

for raw_path in sorted(rbsa_iii_raw_dir.glob("*.csv")):
    raw_data = pd.read_csv(raw_path, dtype=str)
    row_count = len(raw_data)

    for position, column_name in enumerate(raw_data.columns, start=1):
        values = raw_data[column_name].dropna().str.strip()
        values = values[values.ne("")]
        samples = values.drop_duplicates().head(5).tolist()

        inventory_rows.append({
            "file": raw_path.name,
            "column_position": position,
            "column_name": column_name,
            "file_row_count": row_count,
            "non_empty_count": len(values),
            "non_empty_percent": round(100 * len(values) / row_count, 2) if row_count else 0,
            "unique_non_empty_count": values.nunique(),
            "sample_values": " | ".join(samples),
        })

rbsa_iii_raw_inventory = pd.DataFrame(inventory_rows)
inventory_path = rbsa_iii_dir / "RBSA_III_raw_data_inventory.csv"
rbsa_iii_raw_inventory.to_csv(inventory_path, index=False)

print(f"Saved {len(rbsa_iii_raw_inventory):,} column descriptions to {inventory_path}")
rbsa_iii_raw_inventory

NameError: name 'rbsa_iii_raw_dir' is not defined

In [11]:
rbsa_iii_raw_inventory.groupby("file", as_index=False).agg(
    columns=("column_name", "count"),
    rows=("file_row_count", "first"),
).sort_values("file")

,file,columns,rows
0,Appliance_AirCleaner.csv,8,476
1,Appliance_AudioEquipment.csv,12,1045
2,Appliance_Computer.csv,34,3113
3,Appliance_Dehumidifier.csv,15,53
4,Appliance_Dishwasher.csv,14,1916
...,...,...,...
56,Testing_Trueflow.csv,12,415
57,Usage_One_Line.csv,97,2278
58,UtilMeters.csv,5,2845
59,Water.csv,8,13251


## Build RBSA III Model Inputs

Create one `RBSA_III_ModelInput` record for every RBSA III building listed in `HEMS_RBSA_BASE.csv`. Missing or unavailable values are stored as `None`.

In [3]:
from pathlib import Path
import sys

src_candidates = [Path("../src").resolve(), Path("validation/src").resolve()]
src_dir = next(path for path in src_candidates if (path / "rbsa_iii_model_input.py").exists())
rbsa_iii_dir = src_dir / "data" / "HEMS_RBSA" / "RBSA_III"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from rbsa_iii_model_input import (
    RBSA_III_ModelInput,
    build_rbsa_iii_model_inputs,
    export_rbsa_iii_model_inputs,
)

In [4]:
from pathlib import Path
import sys

src_candidates = [Path("../src").resolve(), Path("validation/src").resolve()]
src_dir = next(path for path in src_candidates if (path / "rbsa_iii_model_input.py").exists())
rbsa_iii_dir = src_dir / "data" / "HEMS_RBSA" / "RBSA_III"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from rbsa_iii_model_input import build_rbsa_iii_model_inputs, export_rbsa_iii_model_inputs

model_input_records = build_rbsa_iii_model_inputs(rbsa_iii_dir)

print(f"RBSA III model inputs: {len(model_input_records):,}")
model_input_records[:3]

RBSA III model inputs: 133


[RBSA_III_ModelInput(building_id='SITE_1023', site_id='SITE_1023', conditioned_area_ft2=2700.0, conditioned_volume_ft3=22864.0, average_height_ft=8.46820698629787, conditioned_volume_to_area_ratio_ft=8.46814814814815, total_building_levels=1, footprint_area_ft2=841.0, footprint_perimeter_ft=116.0, window_to_wall_ratio=0.144791666666667, total_wall_u_value_ip=0.0946602571307528, window_u_value_ip=0.38, ach50=6.16, pct_windows_facing_north=12.5, pct_windows_facing_northeast=0.0, pct_windows_facing_east=31.2, pct_windows_facing_southeast=0.0, pct_windows_facing_south=31.2, pct_windows_facing_southwest=0.0, pct_windows_facing_west=25.0, pct_windows_facing_northwest=0.0, foundation_type='>90% Conditioned Basement', primary_heating_system_type='Central Air Source Heat Pump', primary_heating_fuel_type='Electric', primary_cooling_system_type='Central Air Source Heat Pump', primary_water_heater_technology='Electric Resistance', primary_water_heater_fuel_type='Electric'),
 RBSA_III_ModelInput(bu

In [5]:
from pathlib import Path
import sys

src_candidates = [Path("../src").resolve(), Path("validation/src").resolve()]
src_dir = next(path for path in src_candidates if (path / "rbsa_iii_model_input.py").exists())
rbsa_iii_dir = src_dir / "data" / "HEMS_RBSA" / "RBSA_III"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from rbsa_iii_model_input import export_rbsa_iii_model_inputs

model_input_exports = export_rbsa_iii_model_inputs(rbsa_iii_dir)
model_input_exports

{'records': 133,
 'json': WindowsPath('D:/amber_ob/OBGeneration/validation/src/data/HEMS_RBSA/RBSA_III/RBSA_III_model_inputs.json'),
 'csv': WindowsPath('D:/amber_ob/OBGeneration/validation/src/data/HEMS_RBSA/RBSA_III/RBSA_III_model_inputs.csv'),
 'mapping_csv': WindowsPath('D:/amber_ob/OBGeneration/validation/src/data/HEMS_RBSA/RBSA_III/RBSA_III_model_input_source_mapping.csv')}

In [6]:
field_to_inspect = "primary_water_heater_technology"

model_input_df = pd.read_csv(
    rbsa_iii_dir / "RBSA_III_model_inputs.csv",
    dtype=str,
)

unique_value_counts = (
    model_input_df[field_to_inspect]
    .fillna("<None>")
    .value_counts(dropna=False)
    .rename_axis(field_to_inspect)
    .reset_index(name="building_count")
)
pd.set_option("display.max_colwidth", None)
unique_value_counts


,primary_water_heater_technology,building_count
0,Electric Resistance,70
1,Fossil Fuel Non-Condensing,40
2,Electric Heat Pump (Packaged),11
3,Fossil Fuel Condensing,9
4,<None>,2
5,Indirect Water Heater,1


In [9]:
model_input_df[model_input_df[field_to_inspect].isna()]

,building_id,site_id,conditioned_area_ft2,conditioned_volume_ft3,average_height_ft,conditioned_volume_to_area_ratio_ft,total_building_levels,window_to_wall_ratio,total_wall_u_value_ip,window_u_value_ip,ach50,foundation_type,primary_heating_system_type,primary_heating_fuel_type,primary_cooling_system_type
20,SITE_1285,SITE_1285,4750.0,39401.0,8.29485932764289,8.29494736842105,2.0,NaN,NaN,NaN,5.73,>90% Conditioned Basement,Geothermal Heat Pump,Electric,Geothermal Heat Pump
34,SITE_1410,SITE_1410,2100.0,17829.0,8.49020313957866,8.49,2.0,0.0644202791339942,0.114057259367733,NaN,6.45,>90% Crawlspace,Central Air Source Heat Pump,Electric,Central Air Source Heat Pump
38,SITE_1464,SITE_1464,2150.0,16763.0,7.796909128852,7.79674418604651,1.0,NaN,NaN,NaN,12.0,>90% Conditioned Basement,Baseboard,Electric,Window AC
39,SITE_1467,SITE_1467,5000.0,43147.0,8.62940804197754,8.6294,1.0,NaN,NaN,NaN,6.6,>90% Conditioned Basement,Central Air Source Heat Pump,Electric,Central Air Source Heat Pump
44,SITE_1531,SITE_1531,1250.0,11349.0,9.07948437105064,9.0792,NaN,0.0490842314600975,0.103550554961725,NaN,6.67,>90% Slab,Baseboard,Electric,Window AC
46,SITE_154,SITE_154,3100.0,27167.0,8.7634533488192,8.76354838709677,2.0,0.0518612591605596,0.066361070416133,NaN,3.4,>90% Crawlspace,Furnace,Natural gas,Central AC
48,SITE_155,SITE_155,1800.0,13094.0,7.27444208289054,7.27444444444444,2.0,0.0713705936920223,0.119847252232591,NaN,17.35,Other: 1/2 conditioned basement 1/2 garage,Furnace,Electric,No Cooling
52,SITE_1584,SITE_1584,2090.0,16612.0,7.94855014588189,7.94832535885167,2.0,NaN,NaN,NaN,NaN,>90% Unconditioned Basement,Furnace,Natural gas,Central AC
69,SITE_270,SITE_270,1546.0,11009.0,7.12078651685393,7.12095730918499,1.0,0.127625,0.143645204077944,NaN,NaN,>90% Conditioned Basement,Wall Heater,Electric,No Cooling
86,SITE_408,SITE_408,864.0,6653.0,7.7,7.70023148148148,1.0,0.0660734953703704,0.192327950988956,NaN,NaN,>90% Crawlspace,Ductless Mini-split Heat Pump,Electric,Ductless Mini-split Heat Pump
